In [5]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchmetrics

from transformers import AutoTokenizer

In [6]:
if torch.cuda.is_available():
    device = 'cuda'
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Using device: {device}')

Using device: cpu


In [9]:
import os 
import urllib.request

url = "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"
filename = "IMDB-Dataset.csv"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
    print("Downloaded", filename)
else:
    print(filename, "already exists")

Downloaded IMDB-Dataset.csv


In [10]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
text = list(df['review'].str.replace('<br />',''))
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

In [12]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

e:\Data_Projects\Wine_variety_analysis\.conda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Flami\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Example of how to tokenize text:


In [13]:
seq = text[0][:10]
seq

'One of the'

In [14]:
token_ids = tokenizer(seq)['input_ids']
token_ids

[101, 1448, 1104, 1103, 102]

In [15]:
tokenizer.decode(token_ids+[0,0,0])

'[CLS] One of the [SEP] [PAD] [PAD] [PAD]'

# My code bellow: 

In [17]:
import random
from sklearn.model_selection import train_test_split

torch.manual_seed(67)
np.random.seed(67)

# Step 1: Tokenziing every review into token IDs

In [18]:
token_ids = tokenizer(text, add_special_tokens=False)['input_ids']
vocab_size = tokenizer.vocab_size

Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


# Step 2: 90/10 split

In [19]:
train_ids, test_ids, train_labels, test_labels = train_test_split(
    token_ids, labels, test_size=0.1, random_state=42, stratify=labels
)

print(len(train_ids), len(test_ids))

45000 5000


# Step 3: TokenDataset Class

TokenDataset chooses random indexs or random tokens of the documented reviews to show the model across epochs 

In [20]:
class TokenDataset(Dataset):
    def __init__(self, token_ids, labels, max_len=None):
        self.token_ids = token_ids
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.token_ids)

    def __getitem__(self, idx):
        ids = self.token_ids[idx]

        if self.max_len is not None:
            if len(ids) > self.max_len:
                
                start = random.randint(0, len(ids) - self.max_len)
                ids = ids[start : start + self.max_len]
            else:
                ids = ids + [0] * (self.max_len - len(ids))

        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [22]:
max_len = 100
batch_size = 64

train_ds = TokenDataset(train_ids, train_labels, max_len=max_len)
test_ds  = TokenDataset(test_ids,  test_labels,  max_len=None)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False)


# Building the 1D CNN model

In [23]:
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=100, num_layers=3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        layers = []
        in_channels = embed_dim
        for _ in range(num_layers):
            layers.append(nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1))
            layers.append(nn.SiLU())
            in_channels = hidden_dim
        self.convs = nn.Sequential(*layers)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x): # x (B, L) token ID's
        x = self.embedding(x) # (B, L, 100)
        x = x.transpose(1, 2) # (B, 100, L)
        x = self.convs(x)  # (B, 100, L)
        x = x.mean(dim=2) # (B, 100) Global average pooling 
        return self.fc(x) # (B,2) logits

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=100, num_layers=3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.fc(last)